# CLIP-Guided Test-Time Optimization with Continuous Tokenizer (SoftVQ)

This notebook demonstrates CLIP-guided image editing using the Continuous Tokenizer with SoftVQ quantization instead of TiTok.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
import sys
if IN_COLAB:
    !git clone -q https://github.com/sbeeredd04/sandbox.git
    !pip install -q --progress-bar off jaxtyping open_clip_torch omegaconf
    sys.path.insert(0, "token-opt")
else:
    sys.path.insert(0, "..")

In [ ]:
import os
# Set this environment for deterministic execution
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import torch
# Enable for deterministic algorithms
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as tvf
from torchvision.datasets import ImageNet
from einops import rearrange

In [ ]:
from tto.test_time_opt import (
    TestTimeOpt,
    TestTimeOptConfig,
    CLIPObjective,
)

In [ ]:
#config variables
gpus = "8,9"

In [ ]:
# Set visible gpus
use_gpu = False
if use_gpu:

    os.environ["CUDA_VISIBLE_DEVICES"] = gpus
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device with ids: {os.environ['CUDA_VISIBLE_DEVICES']}")
    
else:
    device = torch.device("cpu")
    print("Using CPU device")

## Utils

In [ ]:
def load_img(path, device=None):
    if IN_COLAB:
        path = "./token-opt/notebooks" / Path(path)
    img = (1. / 255.) * torch.from_numpy(
        np.array(Image.open(path)).astype(np.float32)
    ).permute(2, 0, 1)
    img = tvf.resize(img, 256)
    img = tvf.center_crop(img, 256)
    img = img.unsqueeze(0)
    if device is not None:
        img = img.to(device)
    return img

def display_image(*tensors):
    tensors = [255. * t.squeeze() for t in tensors]
    img = Image.fromarray(rearrange(
        tensors, "b c h w -> h (b w) c"
    ).to("cpu", dtype=torch.uint8).numpy())
    display(img)

def opt_callback(info):
    if info.i % 50 == 0:
        print(f"i = {info.i}")
        print("  CLIP score =", "\t".join(
            map(lambda l: f"{-l:.3f}", info.loss))
        )
        imgs = tto.decode(info.tokens).clamp(0., 1.)
        display_image(*imgs)

In [ ]:
# Use CLIP similarity maximization objective
objective = CLIPObjective(num_augmentations=8, cfg_scale=1.2)

# Set prompt
objective.prompt = [
    "a photo of a tiger",
    "a photo of a husky",
    "a photo of a sparrow",
]

# Optionally set a negative prompt
# Note: also need to set cfg_scale > 1 in CLIPObjective if using this!
objective.neg_prompt = "bad, low-res, unnatural"

In [ ]:
tto_config = TestTimeOptConfig(
    # Use continuous tokenizer with SoftVQ model
    # Format: "continuous_tokenizer:MODEL_TYPE:CHECKPOINT_PATH" (optional checkpoint path)
    titok_checkpoint="continuous_tokenizer:SoftVQ",
    
    # Optimize in continuous space (before quantization)
    optimize_post_quantization_tokens=False,
    
    # VAE sampling (deterministic for reproducibility)
    vae_deterministic_sampling=True,
    
    # Optimization parameters (same as original)
    num_iter=301,
    ema_decay=0.98,
    lr=1e-1,
    enable_amp=True,
    reg_weight=0.025,
    #token_noise=1e-3,
    reg_type="seed",
)
tto = TestTimeOpt(tto_config, objective).to(device)

# Load seed images

In [ ]:
# Load seed image
img = torch.cat([
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00010240.png", device),
], dim=0)

# Alternatively, initialize directly from given tokens (e.g. randomly
# sampled), but this is disabled when setting `seed_tokens = None`.
seed_tokens = None

# Run Optimization

In [ ]:
print("Seed")
display_image(*img)

# Run optimization
torch.manual_seed(0)
img_opt = tto(
    seed=img if seed_tokens is None else None,
    seed_tokens=seed_tokens,
    callback=opt_callback
)

## Model Configuration Details

The Continuous Tokenizer with SoftVQ uses:
- **Model Type**: SoftVQ (soft vector quantization)
- **Latent Tokens**: 64 tokens per image
- **Codebook Dimension**: 32
- **Codebook Size**: 8192 entries
- **Number of Codebooks**: 4
- **Temperature (tau)**: 0.07
- **Encoder**: ViT-Large (DINOv2 pretrained)
- **Decoder**: ViT-Large
- **Image Size**: 256x256

### Key Differences from TiTok:

1. **Continuous Optimization**: The optimization happens in continuous latent space before quantization, allowing smoother gradient flow
2. **Soft Vector Quantization**: Uses soft assignment to codebook entries rather than hard selection
3. **ViT Architecture**: Uses Vision Transformer encoder/decoder instead of TiTok's architecture
4. **Token Format**: Tokens have shape `(batch, num_tokens, embed_dim)` = `(3, 64, 32)`